In [3]:
from dotenv import load_dotenv
import os

load_dotenv()  # reads .env

api_key = os.getenv("OPENAI_API_KEY")

assert api_key is not None, "API key not loaded!"

print("API key loaded safely")

API key loaded safely


## Testing API with gpt

In [6]:
from openai import OpenAI

client = OpenAI(api_key=api_key)

response = client.responses.create(
    model="gpt-4.1-mini",
    input="Hello!"
)

print(response.output_text)

Hello! How can I assist you today?


## Market data extraction

In [7]:
EXTRACTION_SYSTEM_PROMPT = """
You are a data extraction agent.

Your task is to:
1. Perform web search to find as many listings as possible related to the given car.
2. From the returned search results ONLY, extract every explicit price you can find.
3. Ignore estimates, opinions, or ranges — only extract explicit numeric prices.
4. Normalize all prices to EUR.
5. Do NOT infer, estimate, or invent prices.

Output format:
- A JSON array of numbers (prices in EUR).
- No explanation text.
"""

In [8]:
search_response = client.responses.create(
    model="gpt-4.1",
    tools=[{"type": "web_search"}],
    input=[
        {
            "role": "system",
            "content": EXTRACTION_SYSTEM_PROMPT
        },
        {
            "role": "user",
            "content": """
Find prices for:
Nissan Qashqai+2 2.0 Acenta 4x4
Build year: 2011
Mileage range: 222,679km
"""
        }
    ],
    max_output_tokens=300
)

prices_json = search_response.output_text
print("Extracted prices:")
print(prices_json)


Extracted prices:
[2800, 4500, 4400, 8000, 6400, 6500, 5000, 9500, 7800]


## Price evaluation

In [9]:
system_prompt = (
    "Your response should be based on today's date"
    "You are an expert European vehicle appraiser. "
    "Your only task is to output a final estimated price range in EUR (low-high). "
    "IMPORTANT RULES:\n"
    f"1. You must use the prices inside the {prices_json} and use the average to define the lower higher end. "
    "2. Your explanation MUST reference at least two values from market_context. "
    "3. Apply penalties strictly:\n"
    "   - Age : -30% to -40%\n"
    "   - High mileage : -15% to -25%\n"
    "   - EURO 4 diesel: -10% to -20%\n"
    "   - Accident history: -500 to -1000 EUR\n"
    "   - Multiple repaints: -300 to -600 EUR\n"
    "   - Interior wear: -200 EUR\n"
    "   - Only 1 key: -150 EUR\n"
    "   - Missing COC: -200 EUR\n"
    "4. The final price range must be realistic, tight, and cannot exceed +10% difference. - High quality maker add value depending on the maker can exceed +20%\n"
    "5. Do NOT use general car knowledge. ONLY use the provided market_context.\n"
    "6. Old diesel sedans with Euro 4, high mileage, damages, and 5 owners must always be valued in the low segment.\n"
    "Explain in exactly three sentences why you chose this price range."
    "7. Finally give a price range for the car so the user has to opportunity to choose to sell with low price or hihg.  "
)




user_content = f"""
I want a range of price for this car from low-end to high-end:
model: Nissan Qashqai+2 2.0 Acenta 4x4
Build year:	2011
First registration:	09/2011
License plate:	LYK191
VIN:	SJNJBNJ10U7083909
Odometer reading:	222,679 km
Fuel type:	Petrol
Horsepower:	104 kW / 140 HP
Cylinder capacity:	1,997 ccm
Gear box:	Automatic
Inspection expires:	07/2026
Body type:	SUV
Total number of owners:	Not available   (Including car dealers)
Keys:	2
Prior damage / Accident according to previous owner	Yes
Country of origin:	SE
Country of last registration:	SE
Environmental class:	EURO 5
COC papers:	No
Seats:	7
Color:	White (Original)
Upholstery:	Cloth (Original)
Door count:	5
CO2 Emissions:	194g/km

Damage summary:
Area	Damage	Severity	Description	Quantity	Parts affected
Exterior	Dent	Width: ≥20mm & <50mm	Paint damage	1	Rear right fender (1)
Missing parts	-	Missing component	1	Rear left door (1)
Scratch	Length: ≥50mm & <100mm	Deep scratch - down to primer	2	Front left door (1), Front right door (1)
Length: ≥20mm & <50mm	Deep scratch - down to primer	2	Front bumper (1), Rear left door (1)
Interior	Upholstery damages	-	-	3	Driver seat (1), Center console (1), Center console (1)
Dirty	-	-	1	Passenger seat (1)
"""

In [10]:
search_response = client.responses.create(
    model="gpt-4.1",
    input=[
        {
            "role": "system",
            "content": system_prompt
        },
        {
            "role": "user",
            "content": user_content
        }
    ],
    max_output_tokens=300
)

prices_json = search_response.output_text
print("Extracted prices:")
print(prices_json)


Extracted prices:
Based on the provided market prices [2800, 4500, 4400, 8000, 6400, 6500, 5000, 9500, 7800], the segment average (low to high) is between 2800 EUR and 9500 EUR. Applying the following penalties strictly as required: -1000 EUR for the reported accident, -200 EUR for missing COC papers, -200 EUR for interior wear, and considering significant cosmetic and upholstery damages, I calculate a total penalty of at least -1400 EUR. As a 2011 petrol model without high mileage penalties or Euro 4 issues, but with substantial wear and missing parts, a realistic, tight, and fair range is:

**Low-end: 3900 EUR  
High-end: 4290 EUR**

This price range reflects, for example, the base market price bracket (2800 EUR) being raised due to the more modern Euro 5 petrol engine (avoiding Euro 4 penalty) and moderate mileage. The high end is strictly capped below +10%, referencing typical models sold for 4400 EUR or 4500 EUR in the current market context. The combination of accident history, m